In [11]:
import numpy as np
import sympy
from hamiltonian import spinless_model, uv_subs
from second_quantization_selfmade import matrix_elements
from sympy.physics.quantum.operatorordering import normal_ordered_form
from sympy.physics.quantum import Dagger
from itertools import product
from pymablock.block_diagonalization import block_diagonalize

In [14]:
def H_effective(H_0_matrix, H_p_matrix, n):
    dim = H_0_matrix.shape[0]

    subspace_indices = np.concatenate((
        np.full(n, 0, dtype=int),
        np.full(dim - n, 1, dtype=int)
    ))

    return block_diagonalize(
        [H_0_matrix, H_p_matrix],
        subspace_indices=subspace_indices
    )


def collect_constant(expr):
    """Collect constant terms in a fermionic expression.

    Parameters
    ==========

    expr : sympy expression

    Returns

    =======

    constant_terms : sympy expression
    """
    expr = normal_ordered_form(expr.expand(), independent=True, recursive_limit=100)
    return sum(
        [
            term
            for term in expr.as_ordered_terms()
            if not term.has(sympy.physics.quantum.Operator)
        ]
    )

def braket_product(braket, ham):
    return collect_constant(braket[0] * ham * Dagger(braket[1]))

def matrix_hamiltonian(H, basis):
    N = len(basis)

    # First attempt (may use multiprocessing)
    flat_matrix = matrix_elements(H, basis)

    # If multiprocessing silently failed (common in notebooks)
    if flat_matrix is None or len(flat_matrix) == 0:
        print("⚠️  Falling back to serial matrix construction")

        flat_matrix = [
            braket_product((bra, ket), ham=H)
            for bra, ket in product(basis, basis)
        ]

    # Hard safety check
    expected = N * N
    if len(flat_matrix) != expected:
        raise RuntimeError(
            f"Matrix construction failed: expected {expected} "
            f"elements, got {len(flat_matrix)}"
        )

    H_matrix = sympy.Matrix(flat_matrix).reshape(N, N)

    # Optional but very useful sanity check
    if not H_matrix.is_hermitian:
        raise ValueError("Hamiltonian is not Hermitian")

    return H_matrix


def effective_model(spins_aligned=False):
    """
    Construct the effective model in the low-energy subspace with mu_L = mu_R=0 and U=0.

    Parameters
    ----------
    spins_aligned : bool, optional
        Whether the spins are aligned or anti-aligned.
    
    Returns
    -------
    H_eff : sympy.Matrix
        The effective hamiltonian.
    symbols : list
        The symbols of the hamiltonian.
    """
    H_0, H_p, symbols, ordered_basis = spinless_model(spins_aligned=spins_aligned)
    print(f"base de estados do espaço em questão: {ordered_basis}")
    H_0_matrix = matrix_hamiltonian(H_0, ordered_basis)
    H_p_matrix = matrix_hamiltonian(H_p, ordered_basis)
    H_tilde, _, _ = H_effective(H_0_matrix, H_p_matrix, 4)

    H_eff = np.sum(H_tilde[0, 0, :3])
    B, U, mu_L, mu_R, mu_M, eps_M, t, alpha_L, alpha_R, Delta, u, v = symbols
    H_eff = sympy.simplify(uv_subs(H_eff.subs({U : 0, mu_L : 0, mu_R : 0}), symbols)) # simplify so Kernel doesn't crash
    H_vacuum = sympy.Matrix(np.diag([H_eff[0,0] for i in range(4)]))
    H_eff = sympy.simplify(H_eff-H_vacuum)[:4, :4] # subtract vacuum
    return H_eff, symbols

In [15]:
H_eff_anti, symbols = effective_model(spins_aligned = False)
B, U, mu_L, mu_R, mu_M, eps_M, t, alpha_L, alpha_R, Delta, u, v = symbols

[c_L, c_R, \gamma_u, \gamma_d, c_L*c_R, c_L*\gamma_u, c_L*\gamma_d, c_R*\gamma_u, c_R*\gamma_d, \gamma_u*\gamma_d, c_L*c_R*\gamma_u, c_L*c_R*\gamma_d, c_L*\gamma_u*\gamma_d, c_R*\gamma_u*\gamma_d, c_L*c_R*\gamma_u*\gamma_d]
base de estados do espaço em questão: [1, c_L*c_R, c_L, c_R, \gamma_u, \gamma_d, c_L*\gamma_u, c_L*\gamma_d, c_R*\gamma_u, c_R*\gamma_d, \gamma_u*\gamma_d, c_L*c_R*\gamma_u, c_L*c_R*\gamma_d, c_L*\gamma_u*\gamma_d, c_R*\gamma_u*\gamma_d, c_L*c_R*\gamma_u*\gamma_d]


In [35]:
sympy.simplify(H_eff_anti)

Matrix([
[                                                 0,                            t**2*cos(alpha_L + alpha_R)/(-B**2 + \mu_M**2 + 1),                                                                                        0,                                                                                       0],
[t**2*cos(alpha_L + alpha_R)/(-B**2 + \mu_M**2 + 1), 2*t**2*(B*sin(alpha_L)**2 - B*sin(alpha_R)**2 - \mu_M)/(-B**2 + \mu_M**2 + 1),                                                                                        0,                                                                                       0],
[                                                 0,                                                                             0,                           t**2*(-2*B*cos(alpha_L)**2 + B - \mu_M)/(-B**2 + \mu_M**2 + 1), I*t**2*(B*sin(alpha_L - alpha_R) - \mu_M*sin(alpha_L + alpha_R))/(-B**2 + \mu_M**2 + 1)],
[                                                 0,   

We see that for anti-spin parallel configration, the presence of $B \neq 0$ and $\alpha_L \neq \alpha_R$ leaves the ECT term non-zero at $\mu_M = 0$. However, if there is no Zeeman in the middle dot or the spin-orbit angles are symmetric, ECT will not shift.

In [36]:
H_eff_para, _ = effective_model(spins_aligned = True)

⚠️  Falling back to serial matrix construction
⚠️  Falling back to serial matrix construction


In [6]:
sympy.simplify(H_eff_para)

Matrix([
[                                                                                                                      0, I*t**2*sqrt(-\mu_M + sqrt(\mu_M**2 + 1))*sqrt(\mu_M + sqrt(\mu_M**2 + 1))*sin(alpha_L + alpha_R)/(B**2 - \mu_M**2 - 1),                                                                                     0,                                                                                     0],
[I*t**2*sqrt(-\mu_M + sqrt(\mu_M**2 + 1))*sqrt(\mu_M + sqrt(\mu_M**2 + 1))*sin(alpha_L + alpha_R)/(-B**2 + \mu_M**2 + 1),                                     2*t**2*(-B*sin(alpha_L)**2 - B*sin(alpha_R)**2 + B - \mu_M)/(-B**2 + \mu_M**2 + 1),                                                                                     0,                                                                                     0],
[                                                                                                                      0,                                    

For spin parallel configration, the presence of $B \neq 0$ ensures that ECT is non-zero at $\mu_M = 0$. Even if spin-orbit symmetry is present, it does not protect ECT against finite Zeeman shifts.